In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install ffmpeg-python torch torchvision


In [ ]:
# %cd /content/drive/MyDrive/process_data

/content/drive/MyDrive/process_data


In [ ]:
import os
import glob
import torch
import ffmpeg
import torchvision.transforms as transforms
from PIL import Image
from pathlib import Path

class VideoProcessor:
    def __init__(self, fps=15, resolution=(224, 224), clip_duration=5, output_dir="output"):
        self.fps = fps
        self.resolution = resolution
        self.clip_duration = clip_duration  # Độ dài mỗi đoạn video (giây)
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

    def extract_and_preprocess_video(self, video_dir, extracted_dir):
        """Trích xuất video stream, resampling FPS và resize"""
        os.makedirs(extracted_dir, exist_ok=True)

        for video_file in glob.glob(f"{video_dir}/*.*"):
            video_name = Path(video_file).stem
            output_file = os.path.join(extracted_dir, f"{video_name}.mp4")

            print(f"🔄 Trích xuất và xử lý video: {video_file} → {output_file}")
            (
                ffmpeg
                .input(video_file)
                .filter("fps", fps=self.fps)
                .filter("scale", self.resolution[0], self.resolution[1])
                .output(output_file, vcodec="h264", pix_fmt="yuv420p", loglevel="quiet")
                .overwrite_output()
                .run()
            )

            if os.path.exists(output_file):
                print(f"✅ Đã xử lý: {output_file}")
            else:
                print(f"❌ Không thể xử lý: {video_file}")

    def split_video(self, src, split_dir):
        """Chia video thành các đoạn `s` giây"""
        os.makedirs(split_dir, exist_ok=True)
        output_pattern = os.path.join(split_dir, "clip_%03d.mp4")

        print(f"🔄 Chia video: {src} → {output_pattern}")
        (
            ffmpeg
            .input(src)
            .output(output_pattern, c="copy", f="segment", segment_time=self.clip_duration, reset_timestamps=1)
            .run(quiet=True, overwrite_output=True)
        )

        clips = os.listdir(split_dir)
        if clips:
            print(f"✅ Đã tạo {len(clips)} clip trong {split_dir}: {clips}")
        else:
            print("❌ Không có clip nào được tạo!")

    def extract_frames(self, src, frame_dir):
        """Trích xuất frames từ video"""
        os.makedirs(frame_dir, exist_ok=True)
        frame_path = os.path.join(frame_dir, "frame_%04d.jpg")

        print(f"🔄 Trích xuất frames từ {src} → {frame_dir}")
        (
            ffmpeg
            .input(src)
            .output(frame_path, vcodec="png", loglevel="quiet")
            .overwrite_output()
            .run()
        )

        frames = os.listdir(frame_dir)
        if frames:
            print(f"✅ Đã trích xuất {len(frames)} frames trong {frame_dir}")
        else:
            print("❌ Không có frames nào được trích xuất!")

    def convert_to_tensor(self, frame_dir, save_path):
        """Chuyển ảnh thành tensor và lưu .pt"""
        transform = transforms.Compose([transforms.ToTensor()])
        frames = []

        for img_path in sorted(glob.glob(f"{frame_dir}/*.jpg")):
            img = Image.open(img_path).convert("RGB")
            frames.append(transform(img))

        if frames:
            video_tensor = torch.stack(frames)  # [T, C, H, W]
            torch.save(video_tensor, save_path)
            print(f"✅ Lưu tensor thành công: {save_path}")
        else:
            print(f"❌ Không có frames trong thư mục {frame_dir}, không thể tạo tensor!")

    def process_videos(self, video_dir):
        """Pipeline xử lý toàn bộ video"""
        extracted_dir = os.path.join(self.output_dir, "processed_videos")
        split_dir = os.path.join(self.output_dir, "split_clips")
        tensor_dir = os.path.join(self.output_dir, "tensor")

        os.makedirs(split_dir, exist_ok=True)
        os.makedirs(tensor_dir, exist_ok=True)

        # Bước 1: Trích xuất & tiền xử lý video
        self.extract_and_preprocess_video(video_dir, extracted_dir)

        # Bước 2: Chia nhỏ video
        for video_file in glob.glob(f"{extracted_dir}/*.mp4"):
            video_name = Path(video_file).stem
            clip_dir = os.path.join(split_dir, video_name)
            self.split_video(video_file, clip_dir)

            # Bước 3: Xử lý từng clip
            for idx, clip_file in enumerate(sorted(glob.glob(f"{clip_dir}/*.mp4"))):
                clip_name = f"{video_name}_{idx:03d}"  # Định dạng: video1_000, video1_001, ...
                frame_dir = os.path.join(self.output_dir, f"frames_{clip_name}")
                save_path = os.path.join(tensor_dir, f"{clip_name}.pt")

                print(f"🔄 Xử lý clip: {clip_file}")
                self.extract_frames(clip_file, frame_dir)
                self.convert_to_tensor(frame_dir, save_path)

        # Kiểm tra cuối cùng
        final_tensors = os.listdir(tensor_dir)
        if final_tensors:
            print(f"✅ Hoàn thành! Tổng số tensor đã lưu: {len(final_tensors)}")
        else:
            print("❌ Không có tensor nào được lưu!")

# Chạy pipeline
video_dir = r"/content/drive/MyDrive/process_data/src/Videos"
if not os.path.exists(video_dir):
    print(f"❌ Thư mục không tồn tại: {video_dir}")
else:
    files = os.listdir(video_dir)
    if not files:
        print(f"❌ Không có file trong thư mục: {video_dir}")
    else:
        print(f"✅ Danh sách file: {files}")

processor = VideoProcessor(clip_duration=5)
processor.process_videos(video_dir)


✅ Danh sách file: ['Bình luận.mp4', 'input.avi']
🔄 Trích xuất và xử lý video: /content/drive/MyDrive/process_data/src/Videos/Bình luận.mp4 → output/processed_videos/Bình luận.mp4
✅ Đã xử lý: output/processed_videos/Bình luận.mp4
🔄 Trích xuất và xử lý video: /content/drive/MyDrive/process_data/src/Videos/input.avi → output/processed_videos/input.mp4
✅ Đã xử lý: output/processed_videos/input.mp4
🔄 Chia video: output/processed_videos/Bình luận.mp4 → output/split_clips/Bình luận/clip_%03d.mp4
✅ Đã tạo 1 clip trong output/split_clips/Bình luận: ['clip_000.mp4']
🔄 Xử lý clip: output/split_clips/Bình luận/clip_000.mp4
🔄 Trích xuất frames từ output/split_clips/Bình luận/clip_000.mp4 → output/frames_Bình luận_000
✅ Đã trích xuất 226 frames trong output/frames_Bình luận_000
✅ Lưu tensor thành công: output/tensor/Bình luận_000.pt
🔄 Chia video: output/processed_videos/input.mp4 → output/split_clips/input/clip_%03d.mp4
✅ Đã tạo 44 clip trong output/split_clips/in